<a href="https://colab.research.google.com/github/pgaraylillo/tutoriales/blob/main/transcribir_audios_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Transcripción de audios de WhatsApp con Whisper

Notebook para transcribir archivos `.opus` (u otros formatos) almacenados en Google Drive usando **faster-whisper**, optimizado para español.

**Recomendación:** activa GPU antes de ejecutar — `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

---


## 1. Instalar dependencias

In [ ]:
!pip install -q faster-whisper
!apt-get -qq install -y ffmpeg > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.2 MB/s eta 0:00:00


## 2. Verificar GPU

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  Sin GPU. El proceso será mucho más lento. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.")

CUDA disponible: True
GPU: Tesla T4


## 3. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Configuración

Ajusta las rutas y opciones según tu caso. La carpeta de entrada debe existir; la de salida se crea sola.


In [ ]:
# === RUTAS ===
INPUT_DIR  = "/content/drive/MyDrive/SINTALIA/google colab"        # carpeta con los .opus
OUTPUT_DIR = "/content/drive/MyDrive/SINTALIA/google colab/transcripciones"

# === EXTENSIONES A PROCESAR ===
EXTENSIONS = (".opus", ".ogg", ".m4a", ".mp3", ".wav")

# === MODELO WHISPER ===
# Opciones: "tiny", "base", "small", "medium", "large-v3"
# Para español chileno: "large-v3" es el más preciso. "medium" es buen balance velocidad/calidad.
MODEL_SIZE = "large-v3"
LANGUAGE   = "es"

# === FORMATO DE SALIDA ===
INCLUDE_TIMESTAMPS = True   # True = incluye marcas de tiempo [00:01.20 → 00:04.50]
SAVE_FORMAT        = "txt"  # "txt" o "md"

# === COMPORTAMIENTO ===
SKIP_EXISTING = True        # no re-transcribir archivos ya procesados


## 5. Cargar modelo Whisper

In [ ]:
from faster_whisper import WhisperModel
import torch

device      = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Cargando modelo {MODEL_SIZE} en {device} ({compute_type})...")
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
print("✅ Modelo listo.")

Cargando modelo large-v3 en cuda (float16)...
✅ Modelo listo.


## 6. Función de transcripción

In [ ]:
from pathlib import Path

def fmt_ts(seconds: float) -> str:
    """Formatea segundos a [MM:SS.cc]."""
    m, s = divmod(seconds, 60)
    return f"{int(m):02d}:{s:05.2f}"

def transcribe_file(audio_path: Path, output_path: Path) -> dict:
    """Transcribe un archivo de audio y guarda el resultado.

    Devuelve un dict con metadata: duración, idioma detectado, n° de segmentos, texto.
    """
    segments, info = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        beam_size=5,
        vad_filter=True,                 # filtra silencios
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    lines, full_text = [], []
    n_segments = 0
    for seg in segments:
        n_segments += 1
        if INCLUDE_TIMESTAMPS:
            lines.append(f"[{fmt_ts(seg.start)} → {fmt_ts(seg.end)}] {seg.text.strip()}")
        else:
            lines.append(seg.text.strip())
        full_text.append(seg.text.strip())

    # Encabezado
    header = []
    if SAVE_FORMAT == "md":
        header.append(f"# {audio_path.name}\n")
        header.append(f"- **Duración:** {info.duration:.1f}s")
        header.append(f"- **Idioma detectado:** {info.language} (prob {info.language_probability:.2f})")
        header.append(f"- **Modelo:** {MODEL_SIZE}\n")
        header.append("---\n")
    else:
        header.append(f"# {audio_path.name}")
        header.append(f"# Duración: {info.duration:.1f}s | Idioma: {info.language} ({info.language_probability:.2f})")
        header.append(f"# Modelo: {MODEL_SIZE}")
        header.append("")

    output_path.write_text("\n".join(header + lines), encoding="utf-8")

    return {
        "file": audio_path.name,
        "duration": info.duration,
        "segments": n_segments,
        "text": " ".join(full_text),
    }


## 7. Procesar todos los audios

In [ ]:
from pathlib import Path
import time

input_dir  = Path(INPUT_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

assert input_dir.exists(), f"No existe la carpeta: {input_dir}"

audios = sorted([p for p in input_dir.iterdir() if p.suffix.lower() in EXTENSIONS])
print(f"📂 Encontrados {len(audios)} archivo(s) en {input_dir}\n")

resultados = []
t0_global = time.time()

for i, audio in enumerate(audios, 1):
    out_file = output_dir / f"{audio.stem}.{SAVE_FORMAT}"

    if SKIP_EXISTING and out_file.exists():
        print(f"[{i}/{len(audios)}] ⏭️  {audio.name} (ya existe)")
        continue

    print(f"[{i}/{len(audios)}] 🎙️  Transcribiendo {audio.name}...", end=" ", flush=True)
    t0 = time.time()
    try:
        info = transcribe_file(audio, out_file)
        dt = time.time() - t0
        rtf = dt / info["duration"] if info["duration"] else 0
        print(f"✅ {info['duration']:.1f}s en {dt:.1f}s (RTF {rtf:.2f}x)")
        resultados.append(info)
    except Exception as e:
        print(f"❌ Error: {e}")

print(f"\n⏱️  Tiempo total: {time.time() - t0_global:.1f}s")
print(f"📁 Transcripciones guardadas en: {output_dir}")


📂 Encontrados 1 archivo(s) en /content/drive/MyDrive/SINTALIA/google colab

[1/1] ⏭️  AUDIO-2026-04-28-17-21-06.m4a (ya existe)

⏱️  Tiempo total: 0.2s
📁 Transcripciones guardadas en: /content/drive/MyDrive/SINTALIA/google colab/transcripciones


## 8. (Opcional) Generar archivo consolidado

Junta todas las transcripciones en un solo archivo, ordenado por nombre. Útil para conversaciones de WhatsApp.


In [ ]:
consolidated = output_dir / "_TODOS_consolidado.md"

with consolidated.open("w", encoding="utf-8") as f:
    f.write("# Transcripciones consolidadas\n\n")
    f.write(f"Total de archivos: {len(audios)}\n\n---\n\n")
    for audio in audios:
        txt_file = output_dir / f"{audio.stem}.{SAVE_FORMAT}"
        if not txt_file.exists():
            continue
        f.write(f"## 🎙️ {audio.name}\n\n")
        f.write(txt_file.read_text(encoding="utf-8"))
        f.write("\n\n---\n\n")

print(f"✅ Consolidado guardado en: {consolidated}")


✅ Consolidado guardado en: /content/drive/MyDrive/SINTALIA/google colab/transcripciones/_TODOS_consolidado.md


## 💡 Notas

- **Velocidad esperada en T4 (Colab gratis) con `large-v3`:** ~5–10x tiempo real (un audio de 30s tarda ~3–6s).
- **Si te quedas sin VRAM** con `large-v3`, baja a `medium` — la diferencia en español es pequeña.
- **VAD filter** (activado por defecto) ignora silencios largos, lo que mejora precisión y velocidad en audios de WhatsApp.
- **Para diarización** (separar quién habla cuándo), reemplaza `faster-whisper` por [`whisperx`](https://github.com/m-bain/whisperX) — agrega dependencia de `pyannote.audio` con token de HuggingFace.
- **Si los audios tienen jerga clínica**, puedes pasar `initial_prompt="contexto médico anestesiología..."` a `model.transcribe()` para sesgar el vocabulario.
